In [9]:
import os
import glob
import pandas as pd
import numpy as np

print("="*80)
print("SCRIPT 1: AUDITORIA DE QUALIDADE - TAXA DE FUGA VISUAL (N=41)")
print("="*80)

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))

relatorio_qualidade = []

for caminho in arquivos:
    nome = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome.upper() else "CONTROLE"
    
    try:
        df = pd.read_csv(caminho)
        
        # Limpeza de formato e conversão numérica
        for col in ['Gaze_X', 'Gaze_Y', 'Pupil_L', 'Pupil_R']:
            if col in df.columns:
                if df[col].dtype == object or df[col].dtype == str:
                    df[col] = df[col].astype(str).str.replace(',', '.')
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df_face = df[df['Fase_Estimulo'] == 'Exposicao_Face'].copy()
        total_frames = len(df_face)
        
        if total_frames > 0:
            # Fuga Real: NaN, Coordenadas Zeradas/Negativas (Fora da Tela)
            mask_fuga = (
                df_face['Gaze_X'].isna() | 
                (df_face['Gaze_X'] <= 0) | 
                df_face['Gaze_Y'].isna() | 
                (df_face['Gaze_Y'] <= 0)
            )
            
            # Se houver coluna de pupila, valida ausência de reflexo pupilar (olho fechado/piscada)
            if 'Pupil_L' in df_face.columns and 'Pupil_R' in df_face.columns:
                mask_pupila_nula = (df_face['Pupil_L'].fillna(0) <= 0) & (df_face['Pupil_R'].fillna(0) <= 0)
                mask_fuga = mask_fuga | mask_pupila_nula
            
            frames_invalidos = mask_fuga.sum()
            fuga_perc = (frames_invalidos / total_frames) * 100.0
        else:
            fuga_perc = 100.0
            
        status = "Ok" if fuga_perc <= 50.0 else ">50% Fuga"
        
        relatorio_qualidade.append({
            'Paciente': nome,
            'Grupo': grupo,
            'Total_Frames_Face': total_frames,
            'Fuga_Visual_%': fuga_perc,
            'Status': status
        })
    except Exception as e:
        print(f"Erro ao auditar '{nome}': {e}")

df_audit = pd.DataFrame(relatorio_qualidade).sort_values(by='Fuga_Visual_%', ascending=False).reset_index(drop=True)

print(df_audit.to_string(index=False, formatters={'Fuga_Visual_%': '{:,.2f}%'.format}))
print("="*80)

# Destaque para os novos sujeitos recuperados
novos = ['T12', 'T15', 'T61']
df_novos = df_audit[df_audit['Paciente'].apply(lambda x: any(n in x for n in novos))]
print("\nAUDITORIA DOS NOVOS SUJEITOS RECUPERADOS:")
print(df_novos.to_string(index=False, formatters={'Fuga_Visual_%': '{:,.2f}%'.format}))

SCRIPT 1: AUDITORIA DE QUALIDADE - TAXA DE FUGA VISUAL (N=41)
        Paciente    Grupo  Total_Frames_Face Fuga_Visual_%    Status
    T12 CONTROLE CONTROLE               8503        99.99% >50% Fuga
T15 TEA MODERADO      TEA               7829        99.76% >50% Fuga
    T61 CONTROLE CONTROLE               7667        99.40% >50% Fuga
   T28 TEA GRAVE      TEA               2314        75.32% >50% Fuga
T21 TEA MODERADO      TEA               4675        36.73%        Ok
    T17 TEA LEVE      TEA               2990        29.83%        Ok
    T24 TEA LEVE      TEA               5866         8.34%        Ok
   T30 TEA GRAVE      TEA               7171         4.95%        Ok
    T41 CONTROLE CONTROLE               6582         3.46%        Ok
    T27 TEA LEVE      TEA               7802         3.08%        Ok
    T32 TEA LEVE      TEA               6458         2.80%        Ok
    T36 CONTROLE CONTROLE               8233         1.98%        Ok
    T37 CONTROLE CONTROLE               8

In [4]:
import os
import glob
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("="*80)
print("SCRIPT 2: EXTRAÇÃO DE SCANPATH DE MARKOV + DELTAS EMOCIONAIS")
print("="*80)

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
PASTA_PROCESSED = '/workspaces/EyeTracking/data/processed/'
os.makedirs(PASTA_PROCESSED, exist_ok=True)

arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
estados = ['Olhos', 'Nariz', 'Boca', 'Fora']
emocoes = ['Raiva', 'Feliz', 'Neutro']

lista_pacientes = []

def discretizar_estado(row):
    # Hierarquia de prioridade anatômica caso haja sobreposição de caixas
    if row.get('AOI_Ambos_Olhos', False): return 'Olhos'
    if row.get('AOI_Nariz', False): return 'Nariz'
    if row.get('AOI_Boca', False): return 'Boca'
    return 'Fora'

for caminho in arquivos:
    nome = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome.upper() else "CONTROLE"
    
    try:
        df = pd.read_csv(caminho)
        
        # Garantir tipo booleano nas AOIs
        for aoi in ['AOI_Ambos_Olhos', 'AOI_Nariz', 'AOI_Boca', 'AOI_Fora']:
            if aoi in df.columns:
                df[aoi] = df[aoi].fillna(False).astype(bool)
            else:
                df[aoi] = False
                
        df_face = df[(df['Fase_Estimulo'] == 'Exposicao_Face') & (df['Tipo_Estimulo'] == 'Humano')].copy()
        df_face['Estado'] = df_face.apply(discretizar_estado, axis=1)
        
        paciente_ft = {'Paciente': nome, 'Grupo': grupo}
        
        # 1. Matrizes de Markov por Emoção
        matrizes_emocao = {}
        for em in emocoes:
            df_em = df_face[df_face['Emocao'] == em]
            
            # Inicializa a matriz 4x4 zerada
            matriz_prob = pd.DataFrame(0.0, index=estados, columns=estados)
            
            if len(df_em) > 1:
                # Comprime observações sequenciais para capturar mudanças efetivas de foco
                mudancas = df_em[df_em['Estado'] != df_em['Estado'].shift(1)]['Estado'].tolist()
                
                if len(mudancas) > 1:
                    trans = pd.DataFrame({'Origem': mudancas[:-1], 'Destino': mudancas[1:]})
                    ct = pd.crosstab(trans['Origem'], trans['Destino'], normalize='index') * 100.0
                    for o in ct.index:
                        for d in ct.columns:
                            matriz_prob.loc[o, d] = ct.loc[o, d]
            
            matrizes_emocao[em] = matriz_prob
            
            # Salva features brutas da emoção
            for o in estados:
                for d in estados:
                    paciente_ft[f"Trans_{o}_{d}_{em}"] = matriz_prob.loc[o, d]

        # 2. Cálculo dos Deltas Emocionais (Reatividade)
        # Delta Raiva vs Neutro (Aversão/Fuga sob provocação)
        m_raiva = matrizes_emocao['Raiva']
        m_neutro = matrizes_emocao['Neutro']
        m_feliz = matrizes_emocao['Feliz']
        
        for o in estados:
            for d in estados:
                paciente_ft[f"Delta_Raiva_Neutro_{o}_{d}"] = m_raiva.loc[o, d] - m_neutro.loc[o, d]
                paciente_ft[f"Delta_Raiva_Feliz_{o}_{d}"] = m_raiva.loc[o, d] - m_feliz.loc[o, d]

        lista_pacientes.append(paciente_ft)
        
    except Exception as e:
        print(f"Erro no processamento de '{nome}': {e}")

df_markov = pd.DataFrame(lista_pacientes).fillna(0.0)
CAMINHO_SAIDA = os.path.join(PASTA_PROCESSED, 'matriz_features_markov.csv')
df_markov.to_csv(CAMINHO_SAIDA, index=False)

print(f"Extração concluída para {len(df_markov)} pacientes.")
print(f"Total de rotas dinâmicas e deltas extraídos: {df_markov.shape[1] - 2}")
print(f"Matriz salva em: {CAMINHO_SAIDA}")

SCRIPT 2: EXTRAÇÃO DE SCANPATH DE MARKOV + DELTAS EMOCIONAIS


Extração concluída para 41 pacientes.
Total de rotas dinâmicas e deltas extraídos: 80
Matriz salva em: /workspaces/EyeTracking/data/processed/matriz_features_markov.csv
